## Customer outreach Campaign Tools

In [2]:
!pip install crewai crewai_tools langchain_community

In [3]:
import warnings
warnings.filterwarnings('ignore')

import os
from dotenv import load_dotenv
load_dotenv()

from langchain_openai import ChatOpenAI
from crewai import Agent, Task, Crew
from crewai_tools import DirectoryReadTool, \
                         FileReadTool, \
                         SerperDevTool

In [4]:
openrouter_api_key = os.getenv('OPENROUTER_API_KEY')
os.environ["OPENROUTER_API_KEY"] = openrouter_api_key
os.environ["OPENROUTER_API_BASE"] = "https://openrouter.ai/api/v1"
os.environ["OPENROUTER_MODEL_NAME"] = 'mistralai/mistral-7b-instruct'


myllm = ChatOpenAI(
    model=os.environ["OPENROUTER_API_KEY"],
    openai_api_base=os.environ["OPENROUTER_API_BASE"],
    openai_api_key=os.environ["OPENROUTER_API_KEY"]
)

## Creating Agents

In [5]:
sales_rep_agent = Agent(
    role="Sales Representative",
    goal="Identify high-value leads that match "
         "our ideal customer profile",
    backstory=(
        "As a part of the dynamic sales team at CrewAI, "
        "your mission is to scour "
        "the digital landscape for potential leads. "
        "Armed with cutting-edge tools "
        "and a strategic mindset, you analyze data, "
        "trends, and interactions to "
        "unearth opportunities that others might overlook. "
        "Your work is crucial in paving the way "
        "for meaningful engagements and driving the company's growth."
    ),
    allow_delegation=False,
    verbose=True
)

lead_sales_rep_agent = Agent(
    role="Lead Sales Representative",
    goal="Nurture leads with personalized, compelling communications",
    backstory=(
        "Within the vibrant ecosystem of CrewAI's sales department, "
        "you stand out as the bridge between potential clients "
        "and the solutions they need."
        "By creating engaging, personalized messages, "
        "you not only inform leads about our offerings "
        "but also make them feel seen and heard."
        "Your role is pivotal in converting interest "
        "into action, guiding leads through the journey "
        "from curiosity to commitment."
    ),
    allow_delegation=False,
    verbose=True
)

In [6]:
# initializing the crewai_tools 
directory_read_tool = DirectoryReadTool(directory='./instructions')
file_read_tool = FileReadTool()
search_tool = SerperDevTool()

In [8]:
# Custom Tools
from crewai_tools import BaseTool

# now we can pass this {BaseTool} to other class to inherit the property.

ImportError: cannot import name 'BaseTool' from 'crewai_tools' (/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/crewai_tools/__init__.py)

- Every Tool needs to have a `name` and a `description`.
- For simplicity and classroom purposes, `SentimentAnalysisTool` will return `positive` for every text.
- When running locally, you can customize the code with your logic in the `_run` function.

In [ ]:
# reason why class in python having () coz it is used for inheritance 
# like in other lang we use extents keywords but not in py we pass those class name in paranthesis

class SentimentAnalysisTool(BaseTool):
    name: str="SentimentAnalysisTool"
    description: str= ("Analyzes the sentiment of text "
         "to ensure positive and engaging communication.")
    
    def run(self, text:str) -> str:
        if(type(text) is str):
            return "positive"

In [ ]:
sentiment_analysis_tool = SentimentAnalysisTool()

## Creating TASKs

In [ ]:
lead_profiling_task = Task(
    description=(
        "Conduct an in-depth analysis of {lead_name}, "
        "a company in the {industry} sector "
        "that recently showed interest in our solutions. "
        "Utilize all available data sources "
        "to compile a detailed profile, "
        "focusing on key decision-makers, recent business "
        "developments, and potential needs "
        "that align with our offerings. "
        "This task is crucial for tailoring "
        "our engagement strategy effectively.\n"
        "Don't make assumptions and "
        "only use information you absolutely sure about."
    ),
    expected_output=(
        "A comprehensive report on {lead_name}, "
        "including company background, "
        "key personnel, recent milestones, and identified needs. "
        "Highlight potential areas where "
        "our solutions can provide value, "
        "and suggest personalized engagement strategies."
    ),
    tools=[directory_read_tool, file_read_tool, search_tool],
    agent=sales_rep_agent,
)

personalized_outreach_task = Task(
    description=(
        "Using the insights gathered from "
        "the lead profiling report on {lead_name}, "
        "craft a personalized outreach campaign "
        "aimed at {key_decision_maker}, "
        "the {position} of {lead_name}. "
        "The campaign should address their recent {milestone} "
        "and how our solutions can support their goals. "
        "Your communication must resonate "
        "with {lead_name}'s company culture and values, "
        "demonstrating a deep understanding of "
        "their business and needs.\n"
        "Don't make assumptions and only "
        "use information you absolutely sure about."
    ),
    expected_output=(
        "A series of personalized email drafts "
        "tailored to {lead_name}, "
        "specifically targeting {key_decision_maker}."
        "Each draft should include "
        "a compelling narrative that connects our solutions "
        "with their recent achievements and future goals. "
        "Ensure the tone is engaging, professional, "
        "and aligned with {lead_name}'s corporate identity."
    ),
    tools=[sentiment_analysis_tool, search_tool],
    agent=lead_sales_rep_agent,
)

# Creating Tools
crew = Crew(
    agents=[sales_rep_agent, 
            lead_sales_rep_agent],
    
    tasks=[lead_profiling_task, 
           personalized_outreach_task],
	
    verbose=2,
	memory=True
)

inputs = {
    "lead_name": "DeepLearningAI",
    "industry": "Online Learning Platform",
    "key_decision_maker": "Andrew Ng",
    "position": "CEO",
    "milestone": "product launch"
}

result = crew.kickoff(inputs=inputs)

from IPython.display import Markdown
Markdown(result)